# Darbas su Meta šeimos modeliais 

## Įvadas 

Ši pamoka apims: 

- Dviejų pagrindinių Meta šeimos modelių tyrinėjimą - Llama 3.1 ir Llama 3.2, taip pat šiuo metu galimus alternatyvius modelius, kurie naudojami kodo pavyzdžiuose (Llama 3.3 ir Llama 4) 
- Supratimą apie kiekvieno modelio naudojimo atvejus ir scenarijus 
- Kodo pavyzdį, rodantį kiekvieno modelio unikalius bruožus 


## Meta šeimos modeliai 

Šioje pamokoje apžvelgsime 2 modelius iš Meta šeimos arba „Llama bandos“ - Llama 3.1 ir Llama 3.2. Veikiantys kodo pavyzdžiai naudoja Llama 3.3 ir Llama 4, kurie yra šiuo metu prieinamos alternatyvos Microsoft Foundry Models kataloge. 

Šie modeliai turi skirtingas variacijas ir yra prieinami [Microsoft Foundry Models kataloge](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Pastaba:** GitHub Models bus nutrauktas 2026 m. liepos pabaigoje. Daugiau informacijos apie [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) naudojimą AI modelių prototipavimui rasite čia.

> **Pastaba:** Šios pamokos metu nagrinėti konkretūs modelių variantai (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) nuo to laiko buvo pašalinti iš Microsoft Foundry Models katalogo. Žemiau pateikti kodo pavyzdžiai naudoja `Llama-3.3-70B-Instruct` ir `Llama-4-Scout-17B-16E-Instruct` kaip šiuo metu prieinamas alternatyvas Microsoft Foundry Models kataloge, todėl jie vis dar veikia šiandien.

Modelių variantai (kaip buvo iš pradžių aptarta šioje pamokoje): 
- Llama 3.1 - 70B Instruct (nebenaudojamas) 
- Llama 3.1 - 405B Instruct (nebenaudojamas) 
- Llama 3.2 - 11B Vision Instruct (nebenaudojamas) 
- Llama 3.2 - 90B Vision Instruct (nebenaudojamas) 

Šiuo metu galimos alternatyvos, naudojamos kodo pavyzdžiuose: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Pastaba: Anksčiau aptarti Llama 3 variantai saugomi tik istoriniais tikslais ir jų nebeįmanoma rasti Microsoft Foundry Models kataloge.*



## Llama 3.1 

Su 405 milijardais parametrų, Llama 3.1 patenka į atvirojo kodo LLM kategoriją. 

Ši versija yra ankstesnės versijos Llama 3 atnaujinimas, siūlantis: 

- Didesnį konteksto langą - 128 tūkst. žetonų prieš 8 tūkst. žetonų 
- Didžiausią išvesties žetonų kiekį - 4096 prieš 2048 
- Geresnę daugakalbę palaikymą - dėl padidėjusio mokymosi žetonų kiekio 

Dėl to Llama 3.1 gali spręsti sudėtingesnes užduotis statant GenAI programas, įskaitant: 
- Vietinis funkcijų kvietimas - galimybę kvieti išorines priemones ir funkcijas už LLM darbo eigos ribų
- Geresnį RAG našumą - dėl didesnio konteksto lango 
- Sintetinių duomenų generavimą - galimybę kurti veiksmingus duomenis užduotims, tokioms kaip tikslinis tobulinimas 



### Vietinių funkcijų kvietimas 

Llama 3.1 buvo patobulintas, kad efektyviau būtų kviečiamos funkcijos ar įrankiai. Jis taip pat turi du įmontuotus įrankius, kuriuos modelis gali atpažinti kaip reikalingus naudoti, remiantis vartotojo pateiktu užklausa. Šie įrankiai yra: 

- **Brave Search** - gali būti naudojamas norint gauti naujausią informaciją, pvz., orų prognozę, atliekant internetinę paiešką 
- **Wolfram Alpha** - gali būti naudojamas sudėtingesniems matematiniams skaičiavimams, todėl nėra būtina rašyti savo funkcijų. 

Taip pat galite sukurti savo pasirinktinius įrankius, kuriuos LLM gali kvieti. 

Žemiau pateiktame kodo pavyzdyje: 

- Mes apibrėžiame turimus įrankius (brave_search, wolfram_alpha) sistemos užklausoje. 
- Išsiunčiame vartotojo prašymą, klausiantį apie orą tam tikrame mieste. 
- LLM atsakys įrankio kvietimu į Brave Search įrankį, kuris atrodys taip `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Pastaba: Šis pavyzdys tik kviečia įrankį, jei norite gauti rezultatus, turėsite susikurti nemokamą paskyrą Brave API puslapyje ir apibrėžti pačią funkciją` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Nors Llama yra didelis kalbos modelis (LLM), viena iš Llama 3.1 trūkumų buvo daugiarūšiškumas. Tai reiškia, gebėjimą naudoti įvairius įvesties tipus, tokius kaip paveikslėliai kaip užuominos, ir pateikti atsakymus. Ši galimybė yra viena iš pagrindinių Llama 3.2 funkcijų. Šios funkcijos taip pat apima: 

- Daugiarūšiškumas - gebėjimas įvertinti tiek teksto, tiek paveikslėlių užuominas 
- Nedidelės ir vidutinės apimties variacijos (11B ir 90B) - suteikia lankstumo diegimo galimybėms, 
- Tik teksto variacijos (1B ir 3B) - tai leidžia modelį naudoti kraštiniuose / mobiliuosiuose įrenginiuose ir užtikrina mažą delsą 

Daugiarūšiškumo palaikymas yra didelis žingsnis atvirojo kodo modelių pasaulyje. Žemiau pateiktas kodo pavyzdys naudoja tiek paveikslėlį, tiek teksto užuominą, kad gautų paveikslėlio analizę. Kadangi Llama 3.2 90B Vision Instruct buvo nutrauktas, pavyzdyje naudojamas `Llama-4-Scout-17B-16E-Instruct`, šiuo metu prieinamas natūraliai daugiarūšiškas alternatyva. 

### Daugiarūšiškumo palaikymas su Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Mokymasis čia nesibaigia, tęsk kelionę

Baigę šią pamoką, peržiūrėkite mūsų [Generatyviosios dirbtinio intelekto mokymosi kolekciją](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst), kad toliau gilintumėte savo žinias apie generatyvųjį DI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
